# Exploration 10: Google's PageRank

**Name:** Your name here

In 1998, there were millions of web pages and no good way to find the important ones. Google's answer was **PageRank**: imagine people clicking from page to page at random, and rank each page by how much of their time ends up there. In this exploration you'll simulate those people on a small internet, turn the simulation into a **matrix**, find out where the simple idea breaks, and fix it the way Google did. The ranking turns out to be a vector you've met before.

We'll use [panchi](https://github.com/Gustavo-Galvao-e-Silva/panchi) and a made-up internet of 8 pages.

**Every problem has three steps:**

- **Predict:** before running any code, write down what you think will happen. Predictions are graded on being *written first*, not on being right.
- **Run:** run the code. Until the final challenge, you only fill in the blanks; the rest of the code is written for you.
- **Explain:** compare what happened with your prediction and explain it in a sentence or two.

**How to complete this exploration**

1. Open the notebook in Google Colab and choose **File → Save a copy in Drive**, then work in that copy. (Skip this if the notebook was handed to you as your own copy, e.g. through Google Classroom.)
2. **Run the setup cell below once before beginning.** Then run the rest of the cells **from top to bottom**. Click a cell and press **Shift + Enter** to run it.
3. Wherever you see `___`, replace it with your answer and run the cell again.
4. Answer the written questions by double-clicking the *Your answer* cells and typing.
5. When you're done, run the **Export to PDF** cell at the very end and submit the PDF.

### Python in 60 seconds

- `x = 5` **stores** the value `5` under the name `x`. From then on, `x` means `5`.
- `print(x)` shows a value below the cell.
- Lines starting with `#` are **comments**. Python ignores them; they're notes for you.
- Python counts from **0**, not 1. The first row of a matrix is row `0`, the second is row `1`, and so on.
- `A @ B` **multiplies** matrices, or a matrix and a vector.
- `[1, 2]` is a **list** of numbers. We'll use lists to type vectors, like `[1, -1]` for $\begin{bmatrix}1\\-1\end{bmatrix}$.
- `"1/2"` in quotes is exactly one half. panchi keeps it as a fraction, so nothing gets rounded.
- If a cell says *"This cell still has a ___ blank"*, there's still something to fill in!
- Errors are normal and nothing breaks: read the last line of the message, fix the cell, and run it again.

## ⚠️ RUN THIS SETUP CELL ONCE BEFORE BEGINNING THE EXPLORATION ⚠️

Click the ▶ button on the cell below. It takes about 30 seconds:

- A window will pop up asking for permission to access your **Google Drive**. Allow it, so the export cell at the end can find your saved notebook.
- Only run it **once**. Run it again only if Colab disconnects or you come back in a new session.

In [ ]:
# @title ⚠️ RUN THIS SETUP CELL ONCE BEFORE BEGINNING THE EXPLORATION ⚠️
# @markdown Only change this if you **renamed** your notebook:
NOTEBOOK_NAME = "exploration_pagerank"  # @param {type:"string"}
NOTEBOOK_NAME = NOTEBOOK_NAME.strip().removesuffix(".ipynb")

import glob
import io
import json
import os
import tokenize
import uuid

# 1. Connect your Google Drive (a permission window will pop up)
from google.colab import drive

drive.mount("/content/drive")
MY_DRIVE = "/content/drive/MyDrive"

# 2. Install panchi
%pip install -q "panchi>=3.2"


# 3. Find your saved copy of this notebook in Drive, even if it's called "Copy of ..."
#    or has your name in front (the most recently saved one wins)
def find_notebook():
    found = []
    for folder in ["Colab Notebooks", "Classroom"]:
        found += glob.glob(f"{MY_DRIVE}/{folder}/**/*{NOTEBOOK_NAME}*.ipynb", recursive=True)
    return max(found, key=os.path.getmtime) if found else None


# 4. Friendly reminder instead of a confusing error when a cell still has a ___ blank
#    (IPython normally uses ___ as a name for an old output, so blanks wouldn't fail)
def _check_blanks(lines):
    try:
        tokens = tokenize.generate_tokens(io.StringIO("".join(lines)).readline)
        has_blank = any(t.type == tokenize.NAME and t.string == "___" for t in tokens)
    except Exception:
        return lines
    if has_blank:
        return ['print("This cell still has a ___ blank. '
                'Replace every ___ with your answer, then run it again.")\n']
    return lines


transformers = get_ipython().input_transformers_cleanup
transformers[:] = [t for t in transformers if getattr(t, "__name__", "") != "_check_blanks"]
transformers.append(_check_blanks)


# 5. The internet we'll simulate: each page lists the pages it links to
import math

import matplotlib.pyplot as plt
import panchi as pan
from matplotlib.patches import FancyArrowPatch
from IPython.display import HTML, display

PAGES = ["Recipe Blog", "Shop", "Forum", "News", "Encyclopedia", "Cat Memes", "Puzzle Game", "Puzzle Wiki"]

WEB = {
    "Recipe Blog": ["Shop", "Cat Memes"],
    "Shop": ["Forum", "Cat Memes"],
    "Forum": ["Recipe Blog", "Cat Memes"],
    "News": ["Encyclopedia"],
    "Encyclopedia": ["Forum", "News", "Cat Memes"],
    "Cat Memes": ["News", "Encyclopedia"],
}

# The Shop starts selling a puzzle game, and the game and its wiki only link to each other
TRAP_WEB = dict(WEB)
TRAP_WEB["Shop"] = ["Forum", "Cat Memes", "Puzzle Game"]
TRAP_WEB["Puzzle Game"] = ["Puzzle Wiki"]
TRAP_WEB["Puzzle Wiki"] = ["Puzzle Game"]

# The News site puts up a paywall and removes all of its links
PAYWALL_WEB = dict(WEB)
PAYWALL_WEB["News"] = []

_POSITIONS = {
    "Recipe Blog": (90, 180), "Shop": (240, 290), "Forum": (240, 70), "News": (600, 205),
    "Encyclopedia": (600, 60), "Cat Memes": (410, 180), "Puzzle Game": (420, 340), "Puzzle Wiki": (600, 340),
}
STEPS = 1000
SURFERS = 500


def link_matrix(web):
    """Column j says where page j's users go: 1/k to each of the k pages it links to."""
    names = list(web)
    rows = [[0] * len(names) for _ in names]
    for j, name in enumerate(names):
        for target in web[name]:
            rows[names.index(target)][j] = pan.Fraction(1, len(web[name]))
    return pan.Matrix(rows)


def _names(n, names=None):
    if names is not None:
        return list(names)
    return PAGES[:n] + [f"Page {i}" for i in range(len(PAGES), n)]


def _layout(names):
    """Fixed spots for the 8 pages, and a new row at the bottom for any extra pages."""
    positions, extra = [], 0
    for name in names:
        if name in _POSITIONS:
            positions.append(_POSITIONS[name])
        else:
            positions.append((90 + 125 * (extra % 5), 450 + 75 * (extra % 2) + 160 * (extra // 5)))
            extra += 1
    height = max(y for _, y in positions) + 75
    return positions, height


def _numbers(v):
    return [float(value) for value in (v.to_list() if hasattr(v, "to_list") else v)]


def _edges(M):
    rows = M.to_list() if hasattr(M, "to_list") else M
    n = len(rows)
    return [[j, i] for j in range(n) for i in range(n) if i != j and rows[i][j] != 0]


def _web_lists(web):
    names = list(web)
    return names, [[names.index(target) for target in web[name]] for name in names]


def _mulberry32(seed):
    """A tiny random number generator that gives the same numbers as the browser's copy."""
    state = seed & 0xFFFFFFFF

    def imul(a, b):
        return (a * b) & 0xFFFFFFFF

    def draw():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = imul(state ^ (state >> 15), state | 1)
        t = ((t + imul(t ^ (t >> 7), t | 61)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return draw


def _surfer_pages(web, step):
    """Where each of the 500 surfers is after this many clicks (-1 means they left)."""
    names, outs = _web_lists(web)
    draw = _mulberry32(1554)
    where = [0] * SURFERS
    for _ in range(step):
        for s in range(SURFERS):
            if where[s] < 0:
                continue
            links = outs[where[s]]
            where[s] = links[int(draw() * len(links))] if links else -1
    return where


def show(x, names=None):
    """Print the share of users on each page."""
    values = _numbers(x)
    for name, value in zip(_names(len(values), names), values):
        print(f"  {name:<13} {100 * value:6.2f}%")
    print(f"  {'Total':<13} {100 * sum(values):6.2f}%")


def ranking(x, names=None):
    """Print the pages from most users to fewest."""
    values = _numbers(x)
    order = sorted(zip(values, _names(len(values), names)), reverse=True)
    for place, (value, name) in enumerate(order, start=1):
        print(f"  {place}. {name:<13} {100 * value:6.2f}%")


def column_sums(M):
    """Add up each column of M."""
    names = _names(M.cols)
    for name, column in zip(names, M.col_vectors):
        total = sum(column.to_list())
        total = round(total, 6) if isinstance(total, float) else total
        print(f"  {name:<13} column adds to {total}")


def check_link_matrix(L):
    """Compare your L with the web's links, one column at a time."""
    right = link_matrix(WEB)
    def same(u, v):
        return all(abs(float(a) - float(b)) < 1e-9 for a, b in zip(u.to_list(), v.to_list()))

    wrong = [PAGES[j] for j in range(6) if not same(L.col_vectors[j], right.col_vectors[j])]
    if wrong:
        print("These columns don't match the links yet:", ", ".join(wrong))
    else:
        print("Your L matches the web. Every column is right!")


_PALETTE = {"bg": "#45423F", "peach": "#FFB592", "cream": "#F2EDE9", "gray": "#8F8984",
            "text": "#DDD6CE", "track": "#5A5652"}


def _radius(share):
    return 10 + 34 * math.sqrt(max(share, 0))


def _draw(ax_graph, ax_bars, names, edges, shares, dots=None):
    positions, height = _layout(names)
    for ax in (ax_graph, ax_bars):
        ax.set_facecolor(_PALETTE["bg"])
    ax_graph.set_xlim(0, 700)
    ax_graph.set_ylim(height, 0)
    ax_graph.set_aspect("equal")
    ax_graph.axis("off")
    radius = [_radius(s) if dots is None else max(18, 2.3 * math.sqrt(s * SURFERS) + 4) for s in shares]
    for j, i in edges:
        (x1, y1), (x2, y2) = positions[j], positions[i]
        length = math.hypot(x2 - x1, y2 - y1)
        ux, uy = (x2 - x1) / length, (y2 - y1) / length
        start = (x1 + ux * (radius[j] + 2), y1 + uy * (radius[j] + 2))
        end = (x2 - ux * (radius[i] + 4), y2 - uy * (radius[i] + 4))
        arrow = FancyArrowPatch(start, end, connectionstyle="arc3,rad=0.12", arrowstyle="-|>",
                                mutation_scale=11, color=_PALETTE["gray"], lw=1.1, shrinkA=0, shrinkB=0, zorder=1)
        ax_graph.add_patch(arrow)
    for k, ((x, y), name) in enumerate(zip(positions, names)):
        if dots is None:
            ax_graph.add_patch(plt.Circle((x, y), radius[k], color=_PALETTE["peach"], zorder=2))
        else:
            ax_graph.add_patch(plt.Circle((x, y), 18, fill=False, color=_PALETTE["gray"], lw=1.2, zorder=2))
        ax_graph.text(x, y + radius[k] + 15, name, color=_PALETTE["cream"], ha="center", fontsize=8.5, zorder=4)
    if dots is not None:
        xs, ys = zip(*dots) if dots else ([], [])
        ax_graph.scatter(xs, ys, s=5, color=_PALETTE["peach"], zorder=3, linewidths=0)
    y = list(range(len(names)))
    ax_bars.barh(y, [100] * len(names), color=_PALETTE["track"], height=0.62)
    ax_bars.barh(y, [100 * s for s in shares], color=_PALETTE["peach"], height=0.62)
    for k, s in enumerate(shares):
        ax_bars.text(101, k, f"{100 * s:.1f}%", va="center", color=_PALETTE["cream"], fontsize=8.5)
    ax_bars.set_yticks(y, names, color=_PALETTE["text"], fontsize=8.5)
    ax_bars.set_xlim(0, 112)
    ax_bars.set_ylim(len(names) - 0.5, -0.5)
    ax_bars.set_xticks([])
    ax_bars.tick_params(length=0)
    for spine in ax_bars.spines.values():
        spine.set_visible(False)


def _figure(n_pages, height):
    graph_h, bars_h = 7 * height / 700, 0.28 * n_pages + 0.3
    fig = plt.figure(figsize=(7, graph_h + bars_h), facecolor=_PALETTE["bg"])
    grid = fig.add_gridspec(2, 1, height_ratios=[graph_h, bars_h], hspace=0.02)
    return fig, fig.add_subplot(grid[0]), fig.add_subplot(grid[1])


def plot_web(M, x, names=None):
    """Draw the web of link matrix M, with node sizes and bars showing the users x."""
    shares = _numbers(x)
    names = _names(len(shares), names)
    fig, ax_graph, ax_bars = _figure(len(names), _layout(names)[1])
    _draw(ax_graph, ax_bars, names, _edges(M), shares)
    plt.show()
    print(f"Total users: {100 * sum(shares):.1f}%")


def plot_surfers(web, step):
    """Draw the 500 surfers after this many clicks."""
    names, outs = _web_lists(web)
    where = _surfer_pages(web, step)
    positions, height = _layout(names)
    dots, seen = [], [0] * len(names)
    for page in where:
        if page < 0:
            continue
        c = seen[page]
        seen[page] += 1
        angle, r = c * 2.39996, 2.3 * math.sqrt(c)
        dots.append((positions[page][0] + r * math.cos(angle), positions[page][1] + r * math.sin(angle)))
    shares = [count / SURFERS for count in seen]
    fig, ax_graph, ax_bars = _figure(len(names), height)
    _draw(ax_graph, ax_bars, names, [[j, i] for j, o in enumerate(outs) for i in o], shares, dots=dots)
    ax_graph.set_title(f"Step {step}", color=_PALETTE["cream"], fontsize=11)
    plt.show()


_WEB_HTML = r"""
<style>
  #BOX { background: #45423F; color: #F2EDE9; color-scheme: dark; font: 14px/1.4 system-ui, sans-serif; width: 700px; max-width: 100%; padding: 14px 16px; box-sizing: border-box; border-radius: 12px; }
  #BOX svg { width: 100%; display: block; }
  #BOX text { fill: #F2EDE9; font-size: 13px; }
  #BOX .bars text { fill: #DDD6CE; }
  #BOX .readout { display: flex; gap: 22px; font-variant-numeric: tabular-nums; margin: 6px 0 4px; }
  #BOX .readout b { color: #FFB592; font-weight: 600; }
  #BOX label { display: flex; gap: 10px; align-items: center; margin: 8px 0; }
  #BOX label span { width: 44px; color: #FFB592; font-weight: 600; }
  #BOX input { flex: 1; accent-color: #FFB592; }
  #BOX output { width: 50px; text-align: right; font-variant-numeric: tabular-nums; }
  #BOX button { background: #5A5652; color: #F2EDE9; border: 0; border-radius: 6px; padding: 3px 10px; font: inherit; cursor: pointer; }
  #BOX button:hover { background: #8F8984; }
  #BOX .dot { transition: transform 0.25s ease-out; }
</style>
<div id="BOX"></div>
<script>
(() => {
  const data = DATA;
  const box = document.getElementById(data.box);
  const n = data.names.length, STEPS = data.steps;
  const pos = data.positions, H = data.height;
  const surfers = data.mode === "surfers";

  // Every step's users: x[k][page]
  let history = [];
  let where = null;
  if (surfers) {
    let a = 1554;
    const draw = () => {
      a |= 0; a = a + 0x6D2B79F5 | 0;
      let t = Math.imul(a ^ a >>> 15, 1 | a);
      t = t + Math.imul(t ^ t >>> 7, 61 | t) ^ t;
      return ((t ^ t >>> 14) >>> 0) / 4294967296;
    };
    where = new Int8Array((STEPS + 1) * data.surfers);
    for (let k = 1; k <= STEPS; k++) {
      for (let s = 0; s < data.surfers; s++) {
        const p = where[(k - 1) * data.surfers + s];
        const links = p < 0 ? [] : data.outs[p];
        where[k * data.surfers + s] = p < 0 ? -1 : links.length ? links[Math.floor(draw() * links.length)] : -1;
      }
    }
    for (let k = 0; k <= STEPS; k++) {
      const x = new Array(n).fill(0);
      for (let s = 0; s < data.surfers; s++) { const p = where[k * data.surfers + s]; if (p >= 0) x[p] += 1 / data.surfers; }
      history.push(x);
    }
  }
  const run = d => {
    history = [data.start.slice()];
    for (let k = 1; k <= STEPS; k++) {
      const x = history[k - 1], total = x.reduce((s, v) => s + v, 0), next = new Array(n).fill(0);
      for (let i = 0; i < n; i++) {
        let s = 0;
        for (let j = 0; j < n; j++) s += data.matrix[i][j] * x[j];
        next[i] = d * s + (1 - d) * total / n;
      }
      history.push(next);
    }
  };
  if (!surfers) run(data.damping ? 0.85 : 1);

  const radius = share => surfers ? Math.max(18, 2.3 * Math.sqrt(share * data.surfers) + 4) : 10 + 34 * Math.sqrt(Math.max(share, 0));
  const rowH = 24, barsTop = H + 10, totalH = barsTop + n * rowH + 6;
  let svg = `<svg viewBox="0 0 700 ${totalH}">`;
  svg += `<defs><marker id="${data.box}m" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0 0L10 5L0 10z" fill="#8F8984"/></marker></defs>`;
  data.edges.forEach(() => { svg += `<path class="edge" fill="none" stroke="#8F8984" stroke-width="1.4" marker-end="url(#${data.box}m)"/>`; });
  data.names.forEach((name, i) => {
    svg += surfers
      ? `<circle class="node" cx="${pos[i][0]}" cy="${pos[i][1]}" r="18" fill="none" stroke="#8F8984" stroke-width="1.5"/>`
      : `<circle class="node" cx="${pos[i][0]}" cy="${pos[i][1]}" fill="#FFB592"/>`;
    svg += `<text class="label" x="${pos[i][0]}" text-anchor="middle">${name}</text>`;
  });
  if (surfers) for (let s = 0; s < data.surfers; s++) svg += `<circle class="dot" r="2.6" fill="#FFB592"/>`;
  svg += '<g class="bars">';
  data.names.forEach((name, i) => {
    const y = barsTop + i * rowH;
    svg += `<text x="120" y="${y + 16}" text-anchor="end">${name}</text>`;
    svg += `<rect x="130" y="${y + 4}" width="480" height="15" rx="3" fill="#5A5652"/>`;
    svg += `<rect class="bar" x="130" y="${y + 4}" height="15" rx="3" fill="#FFB592"/>`;
    svg += `<text class="pct" x="618" y="${y + 16}"></text>`;
  });
  svg += '</g></svg>';
  const damping = data.damping ? `<label><span>d</span><input class="d" type="range" min="0" max="1" step="0.01" value="0.85"><output class="dout"></output></label>` : "";
  box.innerHTML = `${svg}<div class="readout"><div>Step <b class="k"></b></div><div>Total users <b class="total"></b></div><div>Change from last step <b class="change"></b></div></div>
    <label><span>Step</span><button class="play">Play</button><button class="minus">-1</button><input class="step" type="range" min="0" max="${STEPS}" step="1" value="0"><button class="plus">+1</button><output class="kout"></output></label>${damping}`;

  const $ = q => box.querySelector(q), $$ = q => [...box.querySelectorAll(q)];
  const slider = $(".step"), nodes = $$(".node"), labels = $$(".label"), edges = $$(".edge");
  const bars = $$(".bar"), pcts = $$(".pct"), dots = $$(".dot");
  const show = () => {
    const k = Number(slider.value), x = history[k];
    $(".kout").textContent = k; $(".k").textContent = k;
    const total = x.reduce((s, v) => s + v, 0);
    $(".total").textContent = `${(100 * total).toFixed(1)}%`;
    const change = k ? x.reduce((s, v, i) => s + Math.abs(v - history[k - 1][i]), 0) / 2 : 0;
    $(".change").textContent = `${(100 * change).toFixed(2)}%`;
    x.forEach((share, i) => {
      const r = radius(share);
      if (!surfers) nodes[i].setAttribute("r", r);
      labels[i].setAttribute("y", pos[i][1] + r + 16);
      bars[i].setAttribute("width", 480 * Math.min(share, 1));
      pcts[i].textContent = `${(100 * share).toFixed(1)}%`;
    });
    data.edges.forEach(([j, i], e) => {
      const [x1, y1] = pos[j], [x2, y2] = pos[i];
      const dx = x2 - x1, dy = y2 - y1, len = Math.hypot(dx, dy);
      const ux = dx / len, uy = dy / len, bend = 0.12 * len;
      const r1 = radius(x[j]) + 2, r2 = radius(x[i]) + 5;
      const cx = (x1 + x2) / 2 - uy * bend, cy = (y1 + y2) / 2 + ux * bend;
      const a = Math.hypot(cx - x1, cy - y1), b = Math.hypot(x2 - cx, y2 - cy);
      const sx = x1 + (cx - x1) / a * r1, sy = y1 + (cy - y1) / a * r1;
      const ex = x2 - (x2 - cx) / b * r2, ey = y2 - (y2 - cy) / b * r2;
      edges[e].setAttribute("d", `M${sx} ${sy}Q${cx} ${cy} ${ex} ${ey}`);
    });
    if (surfers) {
      const seen = new Array(n).fill(0);
      dots.forEach((dot, s) => {
        const p = where[k * data.surfers + s];
        if (p < 0) { dot.style.opacity = 0; return; }
        const c = seen[p]++, angle = c * 2.39996, r = 2.3 * Math.sqrt(c);
        dot.style.opacity = 1;
        dot.style.transform = `translate(${pos[p][0] + r * Math.cos(angle)}px, ${pos[p][1] + r * Math.sin(angle)}px)`;
      });
    }
  };
  slider.addEventListener("input", show);
  $(".minus").addEventListener("click", () => { slider.value = Number(slider.value) - 1; show(); });
  $(".plus").addEventListener("click", () => { slider.value = Number(slider.value) + 1; show(); });
  let timer = null;
  const stop = () => { clearInterval(timer); timer = null; $(".play").textContent = "Play"; };
  $(".play").addEventListener("click", () => {
    if (timer) return stop();
    if (Number(slider.value) >= STEPS) slider.value = 0;
    $(".play").textContent = "Pause";
    timer = setInterval(() => {
      const k = Number(slider.value);
      if (k >= STEPS) return stop();
      slider.value = Math.min(STEPS, k + Math.max(1, Math.floor(k / 25)));
      show();
    }, 280);
  });
  if (data.damping) {
    const d = $(".d");
    const update = () => { $(".dout").textContent = Number(d.value).toFixed(2); run(Number(d.value)); show(); };
    d.addEventListener("input", update);
    update();
  } else show();
})();
</script>
"""


def _widget(data):
    box = f"web{uuid.uuid4().hex}"
    data = dict(data, box=box, steps=STEPS)
    data["positions"], data["height"] = _layout(data["names"])
    display(HTML(_WEB_HTML.replace("BOX", box).replace("DATA", json.dumps(data))))


def explore_surfers(web):
    """500 surfers start on the Recipe Blog and click a random link every step."""
    names, outs = _web_lists(web)
    edges = [[j, i] for j, o in enumerate(outs) for i in o]
    _widget({"mode": "surfers", "names": names, "outs": outs, "edges": edges, "surfers": SURFERS})


def explore_web(M, damping=False, names=None):
    """Watch x, M x, M^2 x, ... with a step slider. Everyone starts on the Recipe Blog."""
    rows = [[float(v) for v in row] for row in M.to_list()]
    n = len(rows)
    start = [1.0] + [0.0] * (n - 1)
    _widget({"mode": "matrix", "names": _names(n, names), "matrix": rows, "start": start,
             "edges": _edges(rows), "damping": bool(damping)})


if find_notebook():
    print("Setup complete! You can start the exploration now.")
else:
    print("Setup complete, but your saved copy of this notebook wasn't found in Drive.")
    print("Choose File > Save a copy in Drive, then work in (and run setup in) that copy.")

In [ ]:
import panchi as pan

### Your internet and tools

Our internet has these pages, numbered from 0 like everything in Python:

0. Recipe Blog
1. Shop
2. Forum
3. News
4. Encyclopedia
5. Cat Memes

Later, two more pages show up: 6, Puzzle Game, and 7, Puzzle Wiki. In every picture, an arrow from one page to another is a **link**.

- Press play on a **Sliders** cell to get a picture you can move through time. Drag the **Step** slider, use **-1** and **+1**, or press **Play**.
- `plot_web(...)` and `plot_surfers(...)` draw the same picture for one step. Sliders don't show up in the PDF, so after using them, always fill in the cell below them.
- `show(x)` prints the share of users on each page, and `ranking(x)` lists the pages from most users to fewest.

**Do not use `pan.eigen()` anywhere in this exploration.**

## Problem 1. The random surfer

PageRank models a person browsing with no plan, a **random surfer**. Every step, the surfer looks at the links on their page and clicks one of them at random. A page with 2 links sends the surfer to each one half the time.

We'll send **500 surfers** at once. They all start on the Recipe Blog. Each dot in the picture is one surfer.

In [ ]:
plot_web(link_matrix(WEB), [1, 0, 0, 0, 0, 0])  # everyone starts on the Recipe Blog

**Predict:** count the arrows pointing **into** each page. Which page has the most? After many steps, which page do you think will have the most surfers? Why?

*(write your prediction here **before** running the next cell)*

In [ ]:
# @title Sliders { display-mode: "form" }
explore_surfers(WEB)

Drag the slider slowly from 0 to about 30, then all the way to 1000. Find a step where the bars have stopped changing much, and type it below.

In [ ]:
plot_surfers(WEB, ___)  # a step where the bars have settled

**Explain:** which page ended up with the most surfers? Was it the page with the most links pointing into it? Look at **where** each page's links come from, and explain why the winner beats Cat Memes.

*(write your answer here)*

## Problem 2. Surfers as a matrix

Following 500 surfers is slow and a bit random. Instead, let's follow the **share** of surfers on each page: a vector $\vec x$ with one entry per page, adding up to 1 (100%).

If page $j$ has $k$ links, then each step it sends $\frac1k$ of its surfers along each link. So the shares after one step are $L\vec x$, where column $j$ of the **link matrix** $L$ says where page $j$'s surfers go:

- the entry in row $i$ is $\frac1k$ if page $j$ links to page $i$,
- and $0$ if it doesn't.

For example, the Recipe Blog (page 0) links to the Shop (1) and Cat Memes (5), so column 0 of $L$ is $\begin{bmatrix}0 & \tfrac12 & 0 & 0 & 0 & \tfrac12\end{bmatrix}^T$.

### (a) Build $L$

Here are all the links again:

- Recipe Blog: Shop, Cat Memes
- Shop: Forum, Cat Memes
- Forum: Recipe Blog, Cat Memes
- News: Encyclopedia
- Encyclopedia: Forum, News, Cat Memes
- Cat Memes: News, Encyclopedia

It's easier to type one page at a time, so each line below is **one page's column**, and `.T` (the transpose) stands the lines up as columns.

**Predict:** what should every column of $L$ add up to? Why?

*(write your prediction here **before** running the next cell)*

In [ ]:
L = pan.Matrix([
    # to: Recipe Shop   Forum  News   Encyc  Cats
    [0,     "1/2", 0,     0,     0,     "1/2"],  # from Recipe Blog
    [0,     0,     "1/2", 0,     0,     "1/2"],  # from Shop
    [___,   ___,   ___,   ___,   ___,   ___],    # from Forum
    [0,     0,     0,     0,     1,     0],      # from News
    [___,   ___,   ___,   ___,   ___,   ___],    # from Encyclopedia
    [0,     0,     0,     "1/2", "1/2", 0],      # from Cat Memes
]).T

print(L)
column_sums(L)
check_link_matrix(L)

### (b) One step at a time

Everyone starts on the Recipe Blog, so $\vec x_0 = \begin{bmatrix}1 & 0 & 0 & 0 & 0 & 0\end{bmatrix}^T$.

**Predict:** compute $\vec x_1 = L\vec x_0$ and $\vec x_2 = L\vec x_1$ **by hand**. (*Hint:* $L\vec x_0$ is just one column of $L$. Which one?)

*(write your prediction here **before** running the next cell)*

In [ ]:
x0 = pan.Vector([1, 0, 0, 0, 0, 0])

x1 = ___ @ ___
x2 = ___ @ ___

print("x1 =", x1)
print("x2 =", x2)

**Explain:** did your hand computation match? Check that the entries of $\vec x_2$ match the dots in Problem 1 at step 2 (about 25% on each of 4 pages). Why do they match only approximately?

*(write your answer here)*

### (c) A thousand steps

After $k$ steps the shares are $\vec x_k = L^k\vec x_0$. The slider below shows $\vec x_0, \vec x_1, \vec x_2, \dots$ for the matrix you built, with the same start as the dots.

**Predict:** will the bars move like the dots did in Problem 1, or differently? Will they end in the same place?

*(write your prediction here **before** running the next cell)*

In [ ]:
# @title Sliders { display-mode: "form" }
explore_web(L)

In [ ]:
x1000 = L ** ___ @ x0  # the shares after 1000 steps

plot_web(L, x1000)

**Explain:** compare these bars with the dots from Problem 1. What is the same, and what is different? Why are the matrix's bars smooth while the dots wiggle?

*(write your answer here)*

## Problem 3. When the model breaks

Real internets aren't this tidy. Let's change ours in two ways.

### (a) A trap

The Shop starts selling a puzzle game, so it adds a link to the **Puzzle Game** page. The Puzzle Game links to the **Puzzle Wiki**, and the Puzzle Wiki links back to the game, but neither links anywhere else.

`TRAP_WEB` holds the new internet, and `link_matrix(TRAP_WEB)` builds its $8\times 8$ link matrix, the same way you built $L$.

**Predict:** where will the surfers be after 1000 steps? Which page will PageRank call the most important?

*(write your prediction here **before** running the next cell)*

In [ ]:
# @title Sliders { display-mode: "form" }
explore_surfers(TRAP_WEB)

In [ ]:
T = link_matrix(TRAP_WEB)
start = pan.Vector([1, 0, 0, 0, 0, 0, 0, 0])  # everyone on the Recipe Blog

plot_web(T, T ** ___ @ start)  # try 10, then 100, then 1000

**Your results:** what share of the users is on the two puzzle pages after 10, 100 and 1000 steps?

*(write your answer here)*

**Explain:** why do the surfers pile up there? Is the Puzzle Game really the most important page on this internet? What does this say about using $T^k\vec x_0$ to rank pages?

*(write your answer here)*

### (b) A dead end

Back to the original internet. This time the **News** site puts up a paywall and removes all of its links. `PAYWALL_WEB` holds this internet.

**Predict:** what happens to the surfers who land on the News site? What will happen to the **Total users** number?

*(write your prediction here **before** running the next cell)*

In [ ]:
# @title Sliders { display-mode: "form" }
explore_web(link_matrix(PAYWALL_WEB))

In [ ]:
P = link_matrix(PAYWALL_WEB)

print("Column 3 (News):", P.col_vectors[3])
plot_web(P, P ** 10 @ x0)

**Explain:** where did the users go? Use column 3 of $P$ to explain why the total isn't 100% anymore.

*(write your answer here)*

Here's the usual fix: when a surfer reaches a dead end, they type in a random page instead. So the dead end acts as if it **links to every page**, including itself. Fill in the share of the News site's surfers that goes to each of the 6 pages.

In [ ]:
for row in range(6):
    P[row, 3] = ___

column_sums(P)
plot_web(P, P ** 10 @ x0)

**Explain:** why does this fix the dead end? Does it also fix the trap from part (a)? Why or why not?

*(write your answer here)*

## Problem 4. Damping: the bored surfer

Google's fix for traps is the **bored surfer**. Each step, the surfer:

- clicks a random link on their page, with probability $d$ (the **damping factor**),
- or gets bored and jumps to a random page, any of the $n$ pages equally, with probability $1 - d$.

Getting bored sends $\frac{1-d}{n}$ of everyone's surfers to each page. That's the matrix $\frac{1-d}{n}J$, where $J$ is the $n\times n$ matrix of all 1s. Together with the clicks, one step is $\vec x_{k+1} = G\vec x_k$ with the **Google matrix**

$$G = d\,T + \frac{1-d}{n}\,J.$$

Google used $d = 0.85$: 85% clicks, 15% boredom.

### (a) Build $G$

**Predict:** what should the columns of $G$ add up to? Explain why without computing.

*(write your prediction here **before** running the next cell)*

In [ ]:
d = 0.85
n = 8
J = pan.Matrix([[1] * n for _ in range(n)])  # every entry is 1

G = ___ * T + (1 - ___) / n * J

column_sums(G)
plot_web(T, G ** 1000 @ start)

**Explain:** what share of the users is on the two puzzle pages now? Which page is the most important?

*(write your answer here)*

### (b) Turning the dial

The slider below runs $G\vec x_0, G^2\vec x_0, \dots$ for the trap internet, and lets you change $d$ too. Try **at least** $d = 1$, $d = 0$, $d = 0.5$, $d = 0.85$ and $d = 0.99$. For each, move to step 1000, and note:

- the share on the two puzzle pages,
- roughly the first step where **Change from last step** shows 0.00%.

**Predict:** what do you expect at $d = 1$? At $d = 0$?

*(write your prediction here **before** running the next cell)*

In [ ]:
# @title Sliders { display-mode: "form" }
explore_web(T, damping=True)

**Your results:** for each $d$ you tried, list the share on the puzzle pages and the step where the change reached 0.00%.

*(write your answer here)*

**Explain:** what does PageRank measure at $d = 1$, and at $d = 0$? Why is neither useful? Why is a value like $0.85$ a good compromise? Besides the trap, what else gets worse as $d$ gets close to 1?

*(write your answer here)*

## Problem 5. The steady state

After enough steps, the bars stop moving. Call the shares where they stop $\vec x$.

**Predict:** if the bars have stopped moving, what should $G\vec x$ be? Will $\vec x$ change if the surfers start somewhere else?

*(write your prediction here **before** running the next cell)*

In [ ]:
x = G ** 1000 @ start
next_x = G @ x

print("x:")
show(x)
print("G x:")
show(next_x)
print("Length of G x - x:", (next_x - x).magnitude)

Now start everyone on a different page. `pan.unit_vector(8, page)` puts everyone on one page; pick any page number from 0 to 7.

In [ ]:
other_start = pan.unit_vector(8, ___)
other_x = G ** 1000 @ other_start

print("Length of other_x - x:", (other_x - x).magnitude)
ranking(other_x)

**Explain:** the steady state satisfies $G\vec x = \vec x$. What kind of vector is $\vec x$ for the matrix $G$, and what number goes with it? (*Hint:* think back to Exploration 7.) Why doesn't the start matter? Write down the final PageRank ranking.

*(write your answer here)*

## Challenge (Optional): game the ranking

You've just made a page, `"My Page"`, and you want it at the top of PageRank. You can't change anyone else's links, but you can make as many new pages as you like and link them however you want. This is called a **link farm**.

This time there are no blanks: **write the code yourself.**

1. Copy the internet with `my_web = dict(TRAP_WEB)`. Add your page and your extra pages, for example `my_web["My Page"] = ["Encyclopedia"]` and `my_web["Spam 1"] = ["My Page"]`. Every page needs **at least one** link.
2. Build the link matrix with `link_matrix(my_web)`, then the Google matrix with $d = 0.85$, as in Problem 4. Use `n = len(my_web)`.
3. Find the steady state and print it with `ranking(x, my_web)`. You can also draw it with `plot_web(M, x, my_web)`.
4. Try **at least 3** designs. Where should your page's own links point? Try sending them to the Encyclopedia, and try sending them back to your spam pages.
5. Take your best design and try it again with $d = 0.99$.

*Hint:* a vector with a 1 on the Recipe Blog and 0 everywhere else is `pan.unit_vector(n, 0)`.

**Predict:** how many spam pages will you need to reach #1? Will it be easier or harder with $d = 0.99$?

*(write your prediction here **before** running the next cell)*

In [ ]:
# Build your link farm here

In [ ]:
# Try it again with d = 0.99 here

**Your results:** for each design, describe its links and give your page's rank and share.

*(write your answer here)*

**Explain:** which design worked best, and why? How is your best link farm like the puzzle trap from Problem 3? Why does a smaller $d$ limit how much a link farm can gain? If you worked at Google, how might you detect a link farm?

*(write your answer here)*

## Export to PDF

When you've finished everything above:

1. **Save** the notebook (**File → Save**, or Ctrl/Cmd + S). The PDF is made from the *saved* copy in your Drive.
2. Click the ▶ button on the cell below. The first time, it spends a few minutes installing the PDF tools before making your PDF.

Your PDF will download to your computer and also be saved in the main folder of your Google Drive (**My Drive**). The setup and export cells won't show up in the PDF.

In [ ]:
# @title Export to PDF
import shutil

from google.colab import files


def pdf_tools_ready():
    return shutil.which("xelatex") is not None and shutil.which("pandoc") is not None


# Install the PDF tools (LaTeX + pandoc) the first time this runs. It takes a few minutes.
if not pdf_tools_ready():
    print("Installing the PDF tools. This takes a few minutes, but only the first time...")
    !apt-get update -qq > /dev/null
    !DEBIAN_FRONTEND=noninteractive apt-get install -y -qq texlive-xetex texlive-fonts-recommended texlive-latex-extra texlive-plain-generic pandoc > /dev/null

NOTEBOOK = find_notebook()

if not pdf_tools_ready():
    print("Installing the PDF tools failed (see the messages above). Run this cell again.")
elif NOTEBOOK is None:
    print("Couldn't find your saved copy of this notebook in Drive.")
    print("Choose File > Save a copy in Drive, then work in that copy.")
else:
    # Copy the saved notebook out of Drive and convert it here. The PDF shows code and
    # outputs, hides the In [ ]: prompts, and leaves out the setup and export cells.
    LOCAL = f"/content/{NOTEBOOK_NAME}.ipynb"
    PDF = f"/content/{NOTEBOOK_NAME}.pdf"
    !rm -f "{PDF}"
    !cp "{NOTEBOOK}" "{LOCAL}"
    !jupyter nbconvert --to PDF "{LOCAL}" --log-level=ERROR --TemplateExporter.exclude_input_prompt=True --TemplateExporter.exclude_output_prompt=True --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags=remove_cell --TagRemovePreprocessor.remove_input_tags=remove_input

    if os.path.exists(PDF):
        !cp "{PDF}" "{MY_DRIVE}/"
        print("Done! Your PDF is in My Drive and is downloading now:", os.path.basename(PDF))
        files.download(PDF)
    else:
        print("Something went wrong making the PDF; see the messages above.")